In [4]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.metrics import average_precision_score
from sklearn.impute import SimpleImputer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set pandas display options
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### loads `.env`
#### Setting up the `database connection`

In [2]:
load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

engine = create_engine(pg_url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET search_path TO mart, curated, public;"))

with engine.begin() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-12-01 00:06:54.541368-05:00


#### 6.1: Baseline LightGBM with 5-fold CV (PR-AUC + Precision@K)

In [8]:
import lightgbm as lgb

# Load aligned provider train and folds
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)

# Identify ID + label
id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"

assert y_col in df.columns, "Label column not found in mart.provider_train"
assert {"cv_fold", id_col}.issubset(
    folds.columns), "provider_cv_folds missing keys"

# Join folds
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")
assert df["cv_fold"].notna().all(), "Some providers missing cv_fold"

# Features: drop ID/label/fold; numeric only
drop_cols = {id_col, y_col, "cv_fold"}
X = df.drop(columns=list(drop_cols)).apply(pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)

# Impute medians
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

# Class imbalance weighting
pos = int((y == 1).sum())
neg = int((y == 0).sum())
scale_pos_weight = neg / max(1, pos)
print({"n_providers": len(y), "positives": pos, "negatives": neg,
       "scale_pos_weight": round(scale_pos_weight, 2)})

# Metrics


def precision_at_k(y_true, y_score, k):
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))


K_LIST = [50, 100, 200]   # adjust to your review budget

# Model params (conservative baseline)
lgb_params = dict(
    objective="binary",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

# Cross-validation
oof_pred = np.zeros(len(y))
fold_metrics = []

for k in sorted(df["cv_fold"].unique()):
    tr_idx = df.index[df["cv_fold"] != k]
    va_idx = df.index[df["cv_fold"] == k]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_imp.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X_imp.iloc[va_idx], y.iloc[va_idx])],
        # use callbacks for logging + early stopping (version-safe)
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=0)
        ]
    )

    p = model.predict_proba(
        X_imp.iloc[va_idx], num_iteration=model.best_iteration_)[:, 1]
    oof_pred[va_idx] = p

    ap = average_precision_score(y.iloc[va_idx], p)
    p_at_k = {
        f"p@{K}": precision_at_k(y.iloc[va_idx].values, p, K) for K in K_LIST}
    fold_metrics.append({"fold": int(k), "AP": ap, **p_at_k})

# Aggregate metrics
fold_df = pd.DataFrame(fold_metrics)
ap_mean, ap_std = fold_df["AP"].mean(), fold_df["AP"].std()
p_at_k_means = {col: fold_df[col].mean()
                for col in fold_df.columns if col.startswith("p@")}

print("\nFold metrics:")
print(fold_df.to_string(index=False))
print("\nOOF summary:")
print({"AP_mean": round(ap_mean, 4), "AP_std": round(ap_std, 4),
      **{k: round(v, 4) for k, v in p_at_k_means.items()}})

# Persist OOF predictions for audit
oof_df = pd.DataFrame(
    {id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": oof_pred})
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS mart.provider_oof_predictions"))
    oof_df.to_sql("provider_oof_predictions",
                  con=engine, schema="mart", index=False)
print("\nSaved: mart.provider_oof_predictions")

{'n_providers': 5410, 'positives': 506, 'negatives': 4904, 'scale_pos_weight': 9.69}
[LightGBM] [Info] Number of positive: 405, number of negative: 3923
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001930 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5032
[LightGBM] [Info] Number of data points in the train set: 4328, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.093577 -> initscore=-2.270725
[LightGBM] [Info] Start training from score -2.270725
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[23]	valid_0's binary_logloss: 0.207177
[LightGBM] [Info] Number of positive: 405, number of negative: 3923
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001817 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5031
[LightGBM] [Info] Number 

#### 6.2 — LightGBM quick tuning

In [ ]:
import itertools
import random
from sklearn.metrics import average_precision_score

# Load aligned train + folds
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)

id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")

X = df.drop(columns=[id_col, y_col, "cv_fold"]).apply(
    pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

# Class weight
pos = int((y == 1).sum())
neg = int((y == 0).sum())
scale_pos_weight = neg / max(1, pos)


def precision_at_k(y_true, y_score, k):
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))


K_LIST = [50, 100, 200]
fold_ids = sorted(df["cv_fold"].unique())

# Baseline params kept fixed
base = dict(
    objective="binary",
    n_estimators=3000,             # allow early_stopping to pick best_iteration
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

# Small, sensible search space (guided by LightGBM docs)
grid = {
    "learning_rate":   [0.03, 0.05, 0.08],
    "num_leaves":      [15, 31, 63],
    "min_data_in_leaf": [20, 50, 100, 200],
    "feature_fraction": [0.6, 0.8, 1.0],
    "bagging_fraction": [0.6, 0.8, 1.0],
    "lambda_l1":       [0.0, 1.0, 5.0],
    "lambda_l2":       [0.0, 1.0, 5.0],
    "min_gain_to_split": [0.0, 0.1]
}

# Draw a compact randomized set (e.g., 40 tries)
random.seed(42)
keys, vals = zip(*grid.items())
tries = 40
samples = []
while len(samples) < tries:
    samples.append(dict(zip(keys, [random.choice(v) for v in vals])))


def cv_score(params):
    # Merge base + trial
    params = {**base, **params}
    oof = np.zeros(len(y))
    fold_rows = []
    for k in fold_ids:
        tr = df.index[df["cv_fold"] != k]
        va = df.index[df["cv_fold"] == k]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_imp.iloc[tr], y.iloc[tr],
            eval_set=[(X_imp.iloc[va], y.iloc[va])],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=0)
            ]
        )
        p = model.predict_proba(
            X_imp.iloc[va], num_iteration=model.best_iteration_)[:, 1]
        oof[va] = p

        ap = average_precision_score(y.iloc[va], p)
        row = {"fold": int(k), "AP": ap}
        for K in K_LIST:
            row[f"p@{K}"] = precision_at_k(y.iloc[va].values, p, K)
        fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)
    summary = {"AP_mean": fold_df["AP"].mean(), "AP_std": fold_df["AP"].std()}
    for K in K_LIST:
        summary[f"p@{K}"] = fold_df[f"p@{K}"].mean()

    return summary, oof


best = None
best_params = None
best_oof = None

for i, trial in enumerate(samples, 1):
    summary, oof = cv_score(trial)
    score_tuple = (summary["AP_mean"], summary["p@100"])   # tie-break on p@100
    if (best is None) or (score_tuple > (best["AP_mean"], best["p@100"])):
        best, best_params, best_oof = summary, trial, oof
    print(
        f"try {i:02d}/{tries}  AP={summary['AP_mean']:.4f}  p@50={summary['p@50']:.3f}  p@100={summary['p@100']:.3f}")

print("\nBest params:", best_params)
print("Best CV summary:", {k: round(v, 4) for k, v in best.items()})

# Save the winning OOF for audit
oof_df = pd.DataFrame(
    {id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": best_oof})
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS mart.provider_oof_predictions"))
    oof_df.to_sql("provider_oof_predictions",
                  con=engine, schema="mart", index=False, if_exists="replace")
print("\nSaved: mart.provider_oof_predictions (tuned)")

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] lambda_l1 is set=0.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.0
[LightGBM] [Warning] lambda_l2 is set=0.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split